# 4.2.0-Run SPOTlight

**Inés Sentís**

Date of execution

In [ ]:
Sys.Date()

## Libraries

In [ ]:
options(warn = -1, verbose = FALSE)
suppressMessages(suppressWarnings({
library(Matrix)
library(data.table)
library(Seurat)
library(dplyr)
library(SPOTlight)
library(igraph)
library(RColorBrewer)
library(tidyverse)
library(NMF)
}))

## Parameters

In [ ]:
args <- commandArgs(TRUE)

pat <- args[2]
subproj <- args[3]
level <- args[4]

In [ ]:
#here::dr_here(show_reason = TRUE)

source(here::here(glue::glue("{subproj}/st_analysis/misc/paths.R")))
set.seed(0)

## Load Data

In [ ]:
info_cohort <- read_csv(here::here("utils/info_samples.csv"))
info_donor <- info_cohort %>% 
  dplyr::filter(patient == pat)

### Load spatial 10x visium

In [ ]:
rhab_tumor <- readRDS(here::here(glue::glue("{ann}/{robj_dir}/{pat}.rds")))

Make sure there are no rows with all 0

In [ ]:
keep_genes <- data.frame(rowSums(as.matrix(rhab_tumor[['Spatial']]@counts)) != 0)
colnames(keep_genes) <- "logical"
keep_genes <- subset(keep_genes, logical==TRUE)
rhab_tumor[['Spatial']]@counts <- rhab_tumor[['Spatial']]@counts[rownames(keep_genes), ]

Make sure there are no columns with all 0

In [ ]:
keep_spots <- data.frame(colSums(as.matrix(rhab_tumor[['Spatial']]@counts)) != 0)
colnames(keep_spots) <- "logical"
keep_spots <- subset(keep_spots, logical==TRUE)
rhab_tumor[['Spatial']]@counts <- rhab_tumor[['Spatial']]@counts[ ,rownames(keep_spots)]

### Load Reference (paired single cell object)

In [ ]:
ref_obj <- readRDS(here::here(glue::glue("{ann}/{robj_dir}/3.3.0-anno_{pat}.rds")))
DefaultAssay(ref_obj) <- "RNA"

Make sure there are no rows with all 0

In [ ]:
keep_genes <- data.frame(rowSums(as.matrix(ref_obj[['RNA']]@counts)) != 0)
colnames(keep_genes) <- "logical"
keep_genes <- subset(keep_genes, logical==TRUE)
ref_obj[['RNA']]@counts <- ref_obj[['RNA']]@counts[rownames(keep_genes), ]

Make sure there are no columns with all 0

In [ ]:
keep_cells <- data.frame(colSums(as.matrix(ref_obj[['RNA']]@counts)) != 0)
colnames(keep_cells) <- "logical"
keep_cells <- subset(keep_cells, logical==TRUE)
ref_obj[['RNA']]@counts <- ref_obj[['RNA']]@counts[ ,rownames(keep_cells)]

## Load markers 

In [ ]:
m_subproj <- info_donor[info_donor$modality == "multiome", ]$subproject
source(here::here(glue::glue("{m_subproj}/multiome_analysis/misc/paths.R")))
markers <- readr::read_csv(here::here(glue::glue("{clust}/{robj_dir}/level_2/3.3.0-{level}_anno_{pat}.csv")))
head(markers)

In [ ]:
markers <- markers %>% 
  dplyr::filter(p_val_adj<0.05 & avg_log2FC>0.5) %>%
  dplyr::group_by(cluster) %>% 
  arrange(desc(avg_log2FC),.by_group = TRUE)

In [ ]:
markers <- markers[markers$gene %in% rownames(rhab_tumor), ]

## SPOTlight Decomposition

In [ ]:
res <- SPOTlight(
    x = ref_obj[['RNA']]@counts,
    y = rhab_tumor[['Spatial']]@counts,
    groups = Idents(ref_obj),
    mgs = data.frame(markers),
    hvg = VariableFeatures(ref_obj),
    weight_id = "avg_log2FC",
    group_id = "annotation",
    gene_id = "gene")

## Save

In [ ]:
source(here::here(glue::glue("{subproj}/st_analysis/misc/paths.R")))
saveRDS(object = res, file = here::here(glue::glue("{ann}/{robj_dir}/4.2.0-{pat}_{level}_spotlight.rds")))

In [ ]:
sessionInfo()